# Session 3, part 2: scikit-learn, preprocessing and a first linear regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/t-hossein/hands-on-ml-biotech/blob/main/03_ObjectsAndSklearn/02_Sklearn.ipynb)

In part 1 we wrote `MeanPredictor`, a class with `fit` and `predict`. **scikit-learn** (imported as `sklearn`) is a library of models that all follow that same pattern. Today we learn the pattern once, and use it for three purposes:

- splitting data and measuring how wrong a model is,
- preparing data (**preprocessing**),
- fitting a first **linear regression**.

        How `fit` actually finds the line (the normal equation, gradient descent) is the topic of the next session.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "../data/" if Path("../data").exists() else "https://raw.githubusercontent.com/t-hossein/hands-on-ml-biotech/main/data/"

weather = pd.read_csv(DATA + "weather.csv")
weather["time"] = pd.to_datetime(weather["time"].astype(str), format="%Y%m%d%H")
weather["LUZ_wind_peak_in5h"] = weather["LUZ_wind_peak"].shift(-5) # the target: wind peak in Luzern 5 hours later
weather = weather.dropna() # the last 5 hours have no target

bikes = pd.read_csv(DATA + "bikes.csv")

print("weather", weather.shape, "\nbikes", bikes.shape)

weather (25213, 58) 
bikes (17379, 13)


## 1. Every scikit-learn model looks the same

| Step | Code | What happens |
| --- | --- | --- |
| 1. Choose a class | `from sklearn.linear_model import LinearRegression` | |
| 2. Create an object | `model = LinearRegression()` | setting hyperparameters |
| 3. Learn from data | `model.fit(X, y)` | what was learned is stored on the object, in attributes ending with `_` |
| 4. Use it | `model.predict(X_new)` | |

Two conventions for the data:

- **`X`** is a table with one row per sample and one column per feature: shape `(n, p)` (always 2D).
- **`y`** is the target: one value per sample, shape `(n,)`.

![image.png](https://raw.githubusercontent.com/t-hossein/hands-on-ml-biotech/main/03_ObjectsAndSklearn/images/02_Sklearn_cell2_image.png)

![image-2.png](https://raw.githubusercontent.com/t-hossein/hands-on-ml-biotech/main/03_ObjectsAndSklearn/images/02_Sklearn_cell2_image-2.png)

*from Hands-On Machine Learning with Scikit-Learn and PyTorch*

In [2]:
X = weather[["LUZ_pressure"]]
y = weather["LUZ_wind_peak_in5h"]

print(X.shape, y.shape)

(25213, 1) (25213,)


The simplest model in scikit-learn is `DummyRegressor`: it ignores `X` and always predicts the mean of `y`. That is exactly our `MeanPredictor`.

In [8]:
from sklearn.dummy import DummyRegressor

dummy = DummyRegressor(strategy="mean")
dummy.fit(X, y)
dummy.predict(X[:3])

array([13.67223258, 13.67223258, 13.67223258])

## 2. Training set and test set

A model that is judged on the data it learned from can look much better than it is. So we hide part of the data during `fit` and use it only to measure the error:

- **training set**: used by `fit`
- **test set**: used only for evaluation

`train_test_split` picks the test rows at random. `random_state` fixes the random choice, so that everyone gets the same split.

**Be Careful**: Our weather data is a time serie. Shuffling time-series data can lead to *data leakage*.

For data like this, split by time: learn from the past, test on the future.

In [ ]:
is_train = weather["time"] < "2017-01-01"

X_train, y_train = X[is_train], y[is_train]
X_test, y_test = X[~is_train], y[~is_train]

print(len(X_train), "training hours,", len(X_test), "test hours")

16739 training hours, 8474 test hours


## 3. How wrong is a model?

For regression, the usual measure is the **mean squared error** between the true values $y_i$ and the predictions $\hat{y}_i$:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Its square root, the **RMSE**, has the same unit as the target (here km/h), so it is easier to read.

Before any real model, measure the error of the dummy model as a**baseline**.

In [9]:
# compute dummy models' RMSE on the test set

## 4. A first linear regression

Linear regression with one feature predicts with a straight line:

$$\hat{y} = \theta_0 + \theta_1 x$$

- $\theta_1$ is the **slope**, stored in `coef_`
- $\theta_0$ is the **intercept**, stored in `intercept_`

`fit` chooses the line with the smallest MSE on the training set. The code is the same four steps as before: only the class name changes.

In [ ]:
# print coef_ and intercept_

Read the slope with its units: for every extra hPa of pressure in Luzern, the predicted wind peak 5 hours later changes by about -0.3 km/h.

A model with one feature can be drawn. Always do it when you can:

In [ ]:
x_line = pd.DataFrame({"LUZ_pressure": [930, 990]}) # two points are enough to draw a line

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X_test["LUZ_pressure"], y_test, s=5, alpha=0.15, label="test data (2017)")
ax.plot(x_line["LUZ_pressure"], linear_model.predict(x_line), color="red", label="linear regression")
ax.axhline(dummy_model.constant_.item(), color="black", linestyle="--", label="baseline (mean)")
ax.set_xlabel("pressure in Luzern (hPa)")
ax.set_ylabel("wind peak 5 hours later (km/h)")
ax.set_title("One feature")
ax.legend()
plt.show()

In [ ]:
# RMSE of the linear model on the test set

One feature helps a little. With several features the model is no longer a line but the idea is the same, one coefficient per feature:

$$\hat{y} = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \dots + \theta_p x_p$$

For scikit-learn nothing changes: `X` just gets more columns.

In [17]:
luzern = ["LUZ_wind_peak", "LUZ_pressure", "LUZ_temperature", "LUZ_precipitation",
          "LUZ_sunshine_duration", "LUZ_wind_mean", "LUZ_wind_direction"]

X_train, X_test = weather.loc[is_train, luzern], weather.loc[~is_train, luzern]

In [ ]:
# fit a LinearRegression on the 7 Luzern features, print its test RMSE and its coefficients

## 5. Preprocessing

Models need data that is **numerical**, **complete** and often **on comparable scales**. Real tables are none of these. scikit-learn's tools for fixing this are called **transformers**. They follow the same pattern, with `transform` instead of `predict`:

- `fit(X)` learns what is needed from the data (a mean, the list of categories, ...)
- `transform(X)` applies it and returns the new data
- `fit_transform(X)` does both in one call

A small table with all three problems:

In [11]:
samples = pd.DataFrame({
    "age": [34, 51, np.nan, 45, 62, 29],
    "expression": [1200.0, 15.0, 300.0, np.nan, 8000.0, 50.0],
    "tissue": ["liver", "brain", "liver", "blood", "brain", "blood"],
})
samples

,age,expression,tissue
0,34.0,1200.0,liver
1,51.0,15.0,brain
2,NaN,300.0,liver
3,45.0,NaN,blood
4,62.0,8000.0,brain
5,29.0,50.0,blood


### Missing values: `SimpleImputer`

Last session we dropped rows with `dropna` or filled them with `fillna`. `SimpleImputer` fills each column with a statistic of that column, and remembers the value so that new data is filled the same way.

In [14]:
# see statistics_
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
imputer.fit_transform(samples[["age", "expression"]])

array([[  34., 1200.],
       [  51.,   15.],
       [  45.,  300.],
       [  45.,  300.],
       [  62., 8000.],
       [  29.,   50.]])

### Scaling: `StandardScaler`

Standardisation subtracts the mean of a column and divides by its standard deviation:

$$z = \frac{x - \text{mean}}{\text{std}}$$

Afterwards every column has mean 0 and standard deviation 1, whatever its original unit.

In [ ]:
# Look at mean_ and scale_

A transformer is fitted on the **training set only**, and then used to transform both the training set and the test set.

```python
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test) 
```

If the test set takes part in `fit`, information about it leaks into the model, and the test error is no longer honest. This is another example of **data leakage**.

### Categories: `OneHotEncoder`

A model cannot multiply "liver" by a coefficient. Numbering the categories (blood = 0, brain = 1, liver = 2) would be wrong too: it claims that liver is "twice" brain. **One-hot encoding** makes one 0/1 column per category instead.

In [ ]:
# Look at get_feature_names_out()

### Back to the weather:

Scale the 7 Luzern features (fit on the training set only) and fit the model again.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # fit on the training set
X_test_scaled = scaler.transform(X_test) # only transform the test set

linear_scaled = LinearRegression()
linear_scaled.fit(X_train_scaled, y_train)

print("test RMSE, raw features: ", round(rmse_luzern, 2))
print("test RMSE, scaled features:", round(root_mean_squared_error(y_test, linear_scaled.predict(X_test_scaled)), 2))

pd.DataFrame({"raw": linear.coef_, "scaled": linear_scaled.coef_}, index=luzern).round(2)

## 6. Plot first, then preprocess

So far we chose the preprocessing from the *type* of a column: numbers get scaled, categories get one-hot encoded. But two numerical columns can need very different treatment, and a table of numbers hides this. A histogram shows it at a glance.

We plot the **training set only**: choosing the preprocessing is part of building the model, so the test set should not influence it.

![image.png](https://raw.githubusercontent.com/t-hossein/hands-on-ml-biotech/main/03_ObjectsAndSklearn/images/02_Sklearn_cell31_image.png)

Read more: 
- https://medium.com/@myonchev99/log-transformations-and-its-implications-for-linear-regression-f4570de0d69d
- https://scikit-learn.org/stable/auto_examples/compose/plot_transformed_target.html

### ✏️ 

1. Split `bikes` into a training and a test set with `train_test_split` (`test_size=0.2`, `random_state=0`). You can pass the whole DataFrame: `train, test = train_test_split(bikes, ...)`.
2. Compute the baseline RMSE on the test set with a `DummyRegressor`.
3. Fit a `LinearRegression` that uses only `temp`, and compute its test RMSE. What does the slope mean?

In [ ]:
# YOUR CODE HERE

### ✏️ Exercise 2: Two baselines for five-hour wind forecasting

Before fitting a machine-learning model, establish what very simple predictions can already achieve. Here you will compare two no-learning baselines for predicting the Luzern wind peak five hours into the future.
1. Load the weather dataset and construct the five-hour-ahead response vector as follows:
```python
import pandas as pd
weather = pd.read_csv(PATH_TO_YOUR_DATA)
y = weather["LUZ_wind_peak"][5:].values
```

2. Mean baseline. Predict the same value for every observation: the mean of y. Compute and report the mean squared error (MSE).
3. Persistence baseline. Predict that the wind peak five hours from now will be equal to the wind peak observed now: ```weather["LUZ_wind_peak"][:-5].values```. Compute and report the MSE.
4. Compare the two MSE values. Which baseline is better, and by how much? Keep both numbers: any more sophisticated model should improve on these baselines to be useful.
5. The persistence forecast may seem like the obvious choice, yet at a five-hour horizon it performs worse. Explain why. At what forecast horizon would you expect persistence to become the stronger baseline?
6. Compare the MSE of the mean predictor with y.var(). They are identical. Explain why this equality is expected rather than a coincidence.


In [21]:
# YOUR CODE HERE

## 8. All steps in one object

- `ColumnTransformer` applies different transformers to different columns,
- `Pipeline` runs steps one after another and behaves like a single model.

Calling `fit` on the pipeline fits every step on the training data only, so the leakage rule is respected automatically.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric = ["temp", "humidity", "windspeed"]
categorical = ["season", "weather", "hour"]

preprocess = ColumnTransformer([
    ("numeric", StandardScaler(), numeric),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("regression", LinearRegression()),
])